# Tutorial: 7DT Single-Epoch SED Classifier

**Paper:** Gregory S. H. Paek et al. 2026, ApJ, 1001, 198  
**DOI:** https://doi.org/10.3847/1538-4357/ae5229  

This notebook demonstrates how to:
1. Load the pretrained hybrid classifier (XGBoost + Isolation Forest)
2. Run multiclass transient classification on 7DT medium-band SED features
3. Compute the iForest anomaly score
4. Apply the hybrid decision rule to flag kilonova (KN) candidates
5. Reproduce the 2D hybrid decision space plot

The tutorial uses two bundled example datasets:
- **`example_data/sample_non_kn_20.csv`** — 60 non-KN transients (6 classes × 10 samples, 20-filter colors)
- **`data/Feature/Engrave/features_20_color_only.csv`** — 7 epochs of AT 2017gfo (the only confirmed kilonova, VLT/X-shooter)

Both the 20-filter and 40-filter configurations are demonstrated.

## 0. Installation

```bash
# Minimum install for running this notebook
pip install numpy pandas scikit-learn xgboost joblib pyyaml matplotlib seaborn

# Full install (feature engineering, retraining, all research notebooks)
pip install -r requirements.txt
```

Clone and set up:
```bash
git clone https://github.com/SilverRon/7DT-Single-Epoch-SED-Classifier.git
cd 7DT-Single-Epoch-SED-Classifier
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Project root (adjust if running from a different directory)
ROOT = Path('.')

print('Imports OK')

---
## 1. Framework Overview

The hybrid framework combines two models operating in a 2D decision space:

| Axis | Source | Meaning |
|------|--------|---------|
| x: `1 − P_max` | XGBoost multiclass classifier | Low P_max → classifier is uncertain → could be KN |
| y: `P_ano` | Isolation Forest anomaly score | High P_ano → more anomalous → could be KN |

**Decision rule (AND gate):**
```
KN candidate  ↔  (1 − P_max) > thresh_pmax  AND  P_ano > thresh_pano
```

Optimal thresholds from Youden's J statistic (Paek et al. 2026):

| Config | thresh_pmax | thresh_pano |
|--------|------------|------------|
| 20 filters | 0.081 | −0.099 |
| 40 filters | 0.051 | −0.087 |

The `P_ano` score is defined as `−model.decision_function(X)`, so higher values indicate greater anomaly from the non-KN training distribution.

---
## 2. Load Pretrained Models (20-filter)

The 20-filter models use C(20, 2) = **190 pairwise color indices** as input features.

Model files:
- `model/Tune_XGBoost_20/xgboost_7DT.pkl` — XGBoost multiclass classifier
- `model/iForest_20/isolation_forest_base.pkl` — Isolation Forest anomaly detector

In [ ]:
# Load XGBoost (20-filter)
xgb_model_20 = joblib.load(ROOT / 'model/Tune_XGBoost_20/xgboost_7DT.pkl')
feature_names_20 = xgb_model_20.feature_names
print(f'XGBoost 20-filter loaded  |  features: {len(feature_names_20)}')

# Load Isolation Forest (20-filter)
iforest_20 = joblib.load(ROOT / 'model/iForest_20/isolation_forest_base.pkl')
print('iForest 20-filter loaded')

### 2.1 Inspect the class labels

The XGBoost model classifies sources into 8 non-KN classes. KN is detected as an anomaly, not a class.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import yaml

# The model was trained with a LabelEncoder on these 8 classes:
CLASS_NAMES = np.array(['AGN', 'Asteroid', 'Ia', 'II', 'Ibc', 'SLSN', 'SV', 'TDE'])

# Reconstruct label encoder (same alphabetical order used during training)
le = LabelEncoder()
le.fit(CLASS_NAMES)
print('Classes:', le.classes_)
print('Encoded:', dict(zip(le.classes_, le.transform(le.classes_))))

---
## 3. Load Example Data

### 3.1 Non-KN sample (60 transients across 6 classes)

In [ ]:
df_nonkn = pd.read_csv(ROOT / 'example_data/sample_non_kn_20.csv')
print(f'Non-KN sample shape: {df_nonkn.shape}')
print(df_nonkn['Class'].value_counts())
df_nonkn.head(3)

### 3.2 AT 2017gfo — the only confirmed kilonova (7 ENGRAVE epochs)

In [ ]:
df_kn = pd.read_csv(ROOT / 'data/Feature/Engrave/features_20_color_only.csv')
print(f'AT 2017gfo shape: {df_kn.shape}')

# Phase of each epoch post-merger [days]
phases = np.array([1.43, 2.42, 3.41, 4.40, 5.40, 6.40, 7.40])
print(f'Epochs: {phases} days post-merger')
df_kn.head(3)

---
## 4. Prepare Feature Arrays

Extract the 190 color feature columns and fill non-detections (NaN) with the sentinel value **−99**.

NaN values represent non-detections (SNR < 3) and carry astrophysical information (upper limit on flux). XGBoost handles them natively via the sentinel, while the Isolation Forest uses the same sentinel.

In [ ]:
def extract_features(df, feature_names, fill_nan=-99):
    """Return feature matrix and class labels from a feature DataFrame."""
    X = df[feature_names].copy().fillna(fill_nan)
    y = df['Class'].values if 'Class' in df.columns else None
    return X, y

X_nonkn, y_nonkn = extract_features(df_nonkn, feature_names_20)
X_kn,    y_kn    = extract_features(df_kn, feature_names_20)

print(f'Non-KN features: {X_nonkn.shape}')
print(f'AT2017gfo features: {X_kn.shape}')

---
## 5. Step 1 — Multiclass Classification (XGBoost)

The XGBoost model outputs a probability vector over the 8 non-KN classes for each source.  
`P_max` is the highest class probability; `1 − P_max` measures classifier uncertainty.

In [ ]:
def predict_xgboost(model, X, class_names):
    """Run XGBoost prediction; return probability matrix, predicted class, and P_max."""
    dmat = xgb.DMatrix(X)
    proba = model.predict(dmat)  # shape (n_samples, n_classes)
    pred_idx = np.argmax(proba, axis=1)
    pred_class = class_names[pred_idx]
    p_max = proba.max(axis=1)
    return proba, pred_class, p_max

proba_nonkn, pred_nonkn, pmax_nonkn = predict_xgboost(xgb_model_20, X_nonkn, CLASS_NAMES)
proba_kn,    pred_kn,    pmax_kn    = predict_xgboost(xgb_model_20, X_kn,    CLASS_NAMES)

print('Non-KN predictions:')
for cls in np.unique(y_nonkn):
    mask = y_nonkn == cls
    acc  = np.mean(pred_nonkn[mask] == cls)
    print(f'  {cls:<10}: predicted={pd.Series(pred_nonkn[mask]).value_counts().to_dict()}  (accuracy {acc:.0%})')

print('\nAT 2017gfo predictions (phase [day] → predicted class):')
for ph, pc, pm in zip(phases, pred_kn, pmax_kn):
    print(f'  +{ph:.2f}d → {pc:<10} (P_max={pm:.3f})')

### 5.1 Visualise class probabilities for AT 2017gfo

In [ ]:
fig, axes = plt.subplots(1, len(phases), figsize=(18, 3), sharey=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(CLASS_NAMES)))

for ax, ph, prob in zip(axes, phases, proba_kn):
    ax.barh(CLASS_NAMES, prob, color=colors)
    ax.set_title(f'+{ph:.2f}d', fontsize=10)
    ax.set_xlim(0, 1)
    ax.axvline(0.5, ls='--', lw=0.7, color='k')
    ax.tick_params(axis='y', labelsize=8)

axes[0].set_xlabel('Probability')
fig.suptitle('AT 2017gfo — XGBoost class probabilities per epoch', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Step 2 — Anomaly Detection (Isolation Forest)

The iForest was trained on non-KN data only. It outputs a continuous anomaly score.  
We define `P_ano = −model.decision_function(X)`, so **higher values indicate greater anomaly**.

In [ ]:
def anomaly_score(iforest, X):
    """Return P_ano = -decision_function(X); higher = more anomalous."""
    return -iforest.decision_function(X)

pano_nonkn = anomaly_score(iforest_20, X_nonkn)
pano_kn    = anomaly_score(iforest_20, X_kn)

print('Non-KN P_ano  mean={:.4f}  std={:.4f}'.format(pano_nonkn.mean(), pano_nonkn.std()))
print('AT2017gfo P_ano per epoch:')
for ph, pa in zip(phases, pano_kn):
    print(f'  +{ph:.2f}d  P_ano={pa:.4f}')

### 6.1 Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pano_nonkn, bins=20, color='steelblue', alpha=0.7, label='Non-KN (sample)', density=True)
for pa, ph in zip(pano_kn, phases):
    ax.axvline(pa, color='crimson', lw=1.2, alpha=0.8)
ax.axvline(pano_kn[0], color='crimson', lw=1.2, label='AT 2017gfo epochs')
ax.axvline(-0.099, ls='--', color='k', lw=1.5, label='threshold (Youden J)')
ax.set_xlabel('$P_{\\rm ano}$  (higher = more anomalous)')
ax.set_ylabel('Density')
ax.set_title('20-filter  |  Isolation Forest anomaly score distribution')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Step 3 — Hybrid Decision (AND Gate)

A source is flagged as a **KN candidate** when it exceeds both thresholds simultaneously:

```
(1 − P_max) > 0.081   AND   P_ano > −0.099      [20-filter]
(1 − P_max) > 0.051   AND   P_ano > −0.087      [40-filter]
```

This 2D AND gate reduces the false-positive rate from ~50% (iForest alone) to ~9%.

In [ ]:
# Thresholds (Youden's J statistic, Paek et al. 2026, Table 3)
THRESH = {
    20: {'pmax': 0.081, 'pano': -0.099},
    40: {'pmax': 0.051, 'pano': -0.087},
}

def hybrid_decision(one_minus_pmax, p_ano, n_filters=20):
    th = THRESH[n_filters]
    return (one_minus_pmax > th['pmax']) & (p_ano > th['pano'])

# Apply to all data
omp_nonkn = 1 - pmax_nonkn
omp_kn    = 1 - pmax_kn

flag_nonkn = hybrid_decision(omp_nonkn, pano_nonkn)
flag_kn    = hybrid_decision(omp_kn,    pano_kn)

print(f'Non-KN flagged as KN candidate: {flag_nonkn.sum()} / {len(flag_nonkn)}  '
      f'(FPR = {flag_nonkn.mean():.1%})')
print(f'AT 2017gfo flagged as KN candidate:')
for ph, f, pm, pa in zip(phases, flag_kn, omp_kn, pano_kn):
    status = '✓ KN FLAGGED' if f else '✗ missed'
    print(f'  +{ph:.2f}d  1-Pmax={pm:.3f}  P_ano={pa:.4f}  → {status}')

### 7.1 Hybrid 2D decision space plot

In [ ]:
CLASS_COLORS = {
    'AGN': 'gold', 'Asteroid': 'black', 'Ia': 'royalblue',
    'II': 'forestgreen', 'Ibc': 'darkorange', 'SLSN': 'gray',
    'SV': 'cyan', 'TDE': 'purple',
}

fig, ax = plt.subplots(figsize=(7, 7))

# Plot non-KN by class
for cls in np.unique(y_nonkn):
    m = y_nonkn == cls
    ax.scatter(omp_nonkn[m], pano_nonkn[m],
               c=CLASS_COLORS.get(cls, 'grey'), s=40, alpha=0.55, label=cls, zorder=2)

# Plot AT 2017gfo epochs
sc = ax.scatter(omp_kn, pano_kn, c=phases, cmap='plasma',
                s=250, marker='*', ec='k', lw=0.8, zorder=5, label='AT 2017gfo')
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Phase [day]', fontsize=12)

# Threshold lines
th = THRESH[20]
ax.axvline(th['pmax'], ls='--', lw=1.5, color='k', label=f"thresh $1-P_{{\\rm max}}$={th['pmax']}")
ax.axhline(th['pano'], ls=':',  lw=1.5, color='k', label=f"thresh $P_{{\\rm ano}}$={th['pano']}")

# Shade KN-candidate region
ax.fill_betweenx([th['pano'], ax.get_ylim()[1] if ax.get_ylim()[1] > th['pano'] else 0.35],
                 th['pmax'], 1.0,
                 alpha=0.07, color='crimson', label='KN candidate region')

ax.set_xlabel(r'$1 - P_{\rm max}$  (XGBoost uncertainty)', fontsize=13)
ax.set_ylabel(r'$P_{\rm ano}$  (iForest anomaly score)', fontsize=13)
ax.set_title('20-filter  |  Hybrid decision space', fontsize=13)
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

---
## 8. 40-filter Configuration

The 40-filter configuration uses C(40, 2) = **780 pairwise color indices** and achieves macro-F₁ ≈ 0.84 vs. 0.80 for 20 filters.

In [ ]:
# Load 40-filter models
xgb_model_40 = joblib.load(ROOT / 'model/Tune_XGBoost_40/xgboost_7DT.pkl')
iforest_40   = joblib.load(ROOT / 'model/iForest_40/isolation_forest_base.pkl')
feature_names_40 = xgb_model_40.feature_names
print(f'XGBoost 40-filter  |  features: {len(feature_names_40)}')

# Load 40-filter example data
df_nonkn_40 = pd.read_csv(ROOT / 'example_data/sample_non_kn_40.csv')
df_kn_40    = pd.read_csv(ROOT / 'data/Feature/Engrave/features_40_color_only.csv')

X_nonkn_40, y_nonkn_40 = extract_features(df_nonkn_40, feature_names_40)
X_kn_40,    _          = extract_features(df_kn_40,    feature_names_40)

print(f'Non-KN 40-filter: {X_nonkn_40.shape}   AT2017gfo 40-filter: {X_kn_40.shape}')

In [ ]:
# Run full pipeline on 40-filter data
_, _, pmax_nonkn_40 = predict_xgboost(xgb_model_40, X_nonkn_40, CLASS_NAMES)
_, pred_kn_40, pmax_kn_40 = predict_xgboost(xgb_model_40, X_kn_40, CLASS_NAMES)

pano_nonkn_40 = anomaly_score(iforest_40, X_nonkn_40)
pano_kn_40    = anomaly_score(iforest_40, X_kn_40)

omp_nonkn_40 = 1 - pmax_nonkn_40
omp_kn_40    = 1 - pmax_kn_40

flag_nonkn_40 = hybrid_decision(omp_nonkn_40, pano_nonkn_40, n_filters=40)
flag_kn_40    = hybrid_decision(omp_kn_40,    pano_kn_40,    n_filters=40)

print('=== 40-filter results ===')
print(f'Non-KN FPR: {flag_nonkn_40.mean():.1%}')
print(f'AT 2017gfo KN recall: {flag_kn_40.mean():.1%}')
print()
for ph, f, pc in zip(phases, flag_kn_40, pred_kn_40):
    status = '✓ KN FLAGGED' if f else '✗ missed'
    print(f'  +{ph:.2f}d  XGB→{pc:<10} → {status}')

---
## 9. Running on Your Own 7DT Observations

### 9.1 Required input format

Your input should be a table of **medium-band magnitudes** (one row per source, one column per filter):  
`m400, m425, m450, ..., m875`  — magnitudes (AB) or `NaN` if undetected.

### 9.2 Compute color indices (feature engineering)

The models expect pairwise magnitude differences (mX − mY for X < Y). Use the snippet below:

In [ ]:
from itertools import combinations

# 20-filter set
FILTERS_20 = [f'm{w}' for w in range(400, 876, 25)]  # m400, m425, ..., m875

def compute_color_features(phot_df, filters=FILTERS_20):
    """
    Compute all pairwise color indices from a photometry table.

    Parameters
    ----------
    phot_df : pd.DataFrame
        Columns: filter names (e.g. 'm400', 'm425', ...), rows: sources.
        Use NaN for non-detections.

    Returns
    -------
    pd.DataFrame with color columns named 'm400-m425', etc.
    """
    colors = {}
    for fa, fb in combinations(filters, 2):
        col = f'{fa}-{fb}'
        if fa in phot_df.columns and fb in phot_df.columns:
            colors[col] = phot_df[fa] - phot_df[fb]
        else:
            colors[col] = np.nan
    return pd.DataFrame(colors, index=phot_df.index)

# Minimal example: one synthetic source
example_phot = pd.DataFrame({
    'm400': [18.5], 'm425': [18.3], 'm450': [18.1], 'm475': [17.9],
    'm500': [17.8], 'm525': [17.7], 'm550': [17.6], 'm575': [17.5],
    'm600': [17.4], 'm625': [17.3], 'm650': [17.2], 'm675': [np.nan],  # non-detection
    'm700': [np.nan], 'm725': [17.0], 'm750': [16.9], 'm775': [16.8],
    'm800': [16.7], 'm825': [16.6], 'm850': [16.5], 'm875': [16.4],
})

color_features = compute_color_features(example_phot)
print(f'Color features shape: {color_features.shape}')
print('First 5 features:', color_features.iloc[0, :5].to_dict())

In [ ]:
# Run inference on the example source
X_new = color_features[feature_names_20].fillna(-99)

proba_new, pred_new, pmax_new = predict_xgboost(xgb_model_20, X_new, CLASS_NAMES)
pano_new = anomaly_score(iforest_20, X_new)
flag_new = hybrid_decision(1 - pmax_new, pano_new, n_filters=20)

print('=== Inference on example source ===')
print(f'Predicted class : {pred_new[0]}')
print(f'P_max           : {pmax_new[0]:.4f}')
print(f'1 - P_max       : {1-pmax_new[0]:.4f}')
print(f'P_ano           : {pano_new[0]:.4f}')
print(f'KN candidate    : {flag_new[0]}')
print()
print('Class probabilities:')
for cls, p in zip(CLASS_NAMES, proba_new[0]):
    print(f'  {cls:<10}: {p:.4f}')

---
## 10. Summary Table

Collect all results in one place.

In [ ]:
rows = []

# AT 2017gfo — 20-filter
for i, ph in enumerate(phases):
    rows.append(dict(
        label='AT2017gfo', phase=ph, config='20-filter',
        pred_class=pred_kn[i],
        one_minus_pmax=round(omp_kn[i], 4),
        p_ano=round(pano_kn[i], 4),
        kn_flag=bool(flag_kn[i]),
    ))

# AT 2017gfo — 40-filter
for i, ph in enumerate(phases):
    rows.append(dict(
        label='AT2017gfo', phase=ph, config='40-filter',
        pred_class=pred_kn_40[i],
        one_minus_pmax=round(omp_kn_40[i], 4),
        p_ano=round(pano_kn_40[i], 4),
        kn_flag=bool(flag_kn_40[i]),
    ))

summary = pd.DataFrame(rows)
print('AT 2017gfo classification summary:')
print(summary.to_string(index=False))

---
## 11. Next Steps

### Reproducing paper results
The full training and evaluation pipeline is in:
- `notebook/Tune_XGBoost_20.ipynb` / `Tune_XGBoost_40.ipynb` — hyperparameter tuning
- `notebook/Train_iForest.ipynb` — Isolation Forest training
- `notebook/Combine_iForest_and_XGBoost_20.ipynb` / `..._40.ipynb` — hybrid framework evaluation

These notebooks require the full feature tables in `data/Feature/`.

### Generating new synthetic photometry
Use the 7DT-Simulator in `simulator/sdtpy.py`:  
```python
import sys; sys.path.insert(0, 'simulator')
from sdtpy import SDTpy
```
See `test/test_simulator.ipynb` for usage examples.

### Retraining
See `script/Tune_XGBoost_FilterSelection.py` for the documented tuning script with Optuna:  
```bash
python script/Tune_XGBoost_FilterSelection.py --n-trials 100
```

### Citation
If you use this code or the pretrained models, please cite:

```bibtex
@article{Paek2026,
  author  = {Paek, Gregory S. H. and Im, Myungshin and Chang, Seo-Won
             and Choi, Hyeonho and Kim, Ji Hoon},
  title   = {A Hybrid Framework for Kilonova Anomaly Detection Using
              Single-epoch SEDs from the 7-Dimensional Telescope},
  journal = {The Astrophysical Journal},
  year    = {2026},
  volume  = {1001},
  pages   = {198},
  doi     = {10.3847/1538-4357/ae5229}
}
```